# Training Model and Deploy



In [0]:
# Install only if needed in the notebook environment
# !pip install -U sagemaker boto3 pandas scikit-learn pyarrow

import os
import io
import json
import time
import uuid
import boto3
import sagemaker
import numpy as np
import pandas as pd

from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Basic SageMaker / AWS configuration
session = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

bucket = "finalproject-fraud-detection"

# S3 prefixes
raw_prefix = "raw"
processed_prefix = "processed"
model_prefix = "model"

# Input raw file in S3
raw_file_name = "PS_20174392719_1491204439457_log.csv"
raw_s3_uri = f"s3://{bucket}/{raw_prefix}/{raw_file_name}"

# Output locations
processed_train_s3_uri = f"s3://{bucket}/{processed_prefix}/train/train.csv"
processed_validation_s3_uri = f"s3://{bucket}/{processed_prefix}/validation/validation.csv"
processed_test_s3_uri = f"s3://{bucket}/{processed_prefix}/test/test.csv"
model_s3_output = f"s3://{bucket}/{model_prefix}/"

# Local temp paths
local_dir = "/tmp/fraud_detection"
os.makedirs(local_dir, exist_ok=True)

train_local = os.path.join(local_dir, "train.csv")
validation_local = os.path.join(local_dir, "validation.csv")
test_local = os.path.join(local_dir, "test.csv")

print("Region:", region)
print("Role:", role)
print("Raw data:", raw_s3_uri)
print("Model output:", model_s3_output)


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Region: us-east-1
Role: arn:aws:iam::998754722171:role/service-role/AmazonSageMakerAdminIAMExecutionRole_1
Raw data: s3://finalproject-fraud-detection/raw/PS_20174392719_1491204439457_log.csv
Model output: s3://finalproject-fraud-detection/model/


In [0]:
# Download the raw CSV file from S3 to local notebook storage
raw_s3_key = f"{RAW_PREFIX}/{RAW_FILE}"
print(f"Downloading s3://{BUCKET}/{raw_s3_key} ...")

s3_client.download_file(BUCKET, raw_s3_key, LOCAL_RAW_FILE)

print("Downloaded to:", LOCAL_RAW_FILE)
print("File size (MB):", round(os.path.getsize(LOCAL_RAW_FILE) / 1024 / 1024, 2))


Downloaded to: /tmp/transactions.csv
File size (MB): 470.67


In [0]:
# This cell uses chunk-based reading to reduce memory usage.
# Strategy:
# 1. Read only needed columns
# 2. Convert numeric columns to smaller dtypes
# 3. One-hot encode transaction type
# 4. Keep all fraud samples
# 5. Randomly downsample non-fraud samples to reduce memory and training cost

USECOLS = [
    "step", "type", "amount",
    "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest",
    "isFraud"
]

DTYPES = {
    "step": "int32",
    "type": "category",
    "amount": "float32",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8"
}

# Keep all fraud rows, and keep only a fraction of non-fraud rows
NON_FRAUD_SAMPLE_FRAC = 0.03
CHUNK_SIZE = 300000
RANDOM_STATE = 42

parts = []
fraud_count = 0
non_fraud_count = 0

for i, chunk in enumerate(pd.read_csv(
    LOCAL_RAW_FILE,
    usecols=USECOLS,
    dtype=DTYPES,
    chunksize=CHUNK_SIZE
)):
    # One-hot encode 'type'
    chunk = pd.get_dummies(chunk, columns=["type"], prefix="type", dtype="int8")

    # Ensure all expected transaction type columns exist
    expected_type_cols = [
        "type_CASH_IN",
        "type_CASH_OUT",
        "type_DEBIT",
        "type_PAYMENT",
        "type_TRANSFER"
    ]
    for col in expected_type_cols:
        if col not in chunk.columns:
            chunk[col] = 0

    # Reorder columns for consistency
    chunk = chunk[
        [
            "step", "amount",
            "oldbalanceOrg", "newbalanceOrig",
            "oldbalanceDest", "newbalanceDest",
            "type_CASH_IN", "type_CASH_OUT", "type_DEBIT",
            "type_PAYMENT", "type_TRANSFER",
            "isFraud"
        ]
    ]

    fraud_part = chunk[chunk["isFraud"] == 1]
    non_fraud_part = chunk[chunk["isFraud"] == 0].sample(
        frac=NON_FRAUD_SAMPLE_FRAC,
        random_state=RANDOM_STATE
    )

    fraud_count += len(fraud_part)
    non_fraud_count += len(non_fraud_part)

    sampled_chunk = pd.concat([fraud_part, non_fraud_part], axis=0)
    parts.append(sampled_chunk)

    print(
        f"Processed chunk {i + 1} | "
        f"kept fraud={len(fraud_part)} | kept non-fraud={len(non_fraud_part)}"
    )

df = pd.concat(parts, ignore_index=True)

# Shuffle the sampled dataset
df = df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nFinal sampled dataset shape:", df.shape)
print("Fraud count kept:", fraud_count)
print("Non-fraud count kept:", non_fraud_count)
print("Fraud ratio:", round(df["isFraud"].mean(), 4))
df.head()


Processed chunk 1 | kept fraud=181 | kept non-fraud=8995


Processed chunk 2 | kept fraud=180 | kept non-fraud=8995


Processed chunk 3 | kept fraud=137 | kept non-fraud=8996


Processed chunk 4 | kept fraud=1020 | kept non-fraud=8969


Processed chunk 5 | kept fraud=90 | kept non-fraud=8997


...

Processed chunk 18 | kept fraud=236 | kept non-fraud=8993


Processed chunk 19 | kept fraud=228 | kept non-fraud=8993


Processed chunk 20 | kept fraud=390 | kept non-fraud=8988


Processed chunk 21 | kept fraud=2686 | kept non-fraud=8919
Processed chunk 22 | kept fraud=712 | kept non-fraud=1857

Final sampled dataset shape: (198844, 12)
Fraud count kept: 8213
Non-fraud count kept: 190631
Fraud ratio: 0.0413


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,isFraud
0,397,27855.000000,2.133400e+04,0.000000e+00,0.000000e+00,0.000000e+00,0,0,0,1,0,0
1,187,3296.719971,3.438677e+05,3.405710e+05,0.000000e+00,0.000000e+00,0,0,0,1,0,0
2,182,25775.500000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0,0,0,1,0,0
3,206,157234.312500,1.551980e+05,0.000000e+00,3.967881e+05,5.540224e+05,0,1,0,0,0,0
4,139,138127.093750,4.483786e+06,4.621914e+06,3.172793e+06,3.034666e+06,1,0,0,0,0,0


In [0]:
# Split dataset into train / validation / test
# SageMaker built-in XGBoost expects label in the first column for CSV input

X = df.drop(columns=["isFraud"])
y = df["isFraud"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

train_df = pd.concat([y_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
valid_df = pd.concat([y_valid.reset_index(drop=True), X_valid.reset_index(drop=True)], axis=1)
test_df  = pd.concat([y_test.reset_index(drop=True),  X_test.reset_index(drop=True)], axis=1)

# Save local CSV files without headers for SageMaker XGBoost
train_df.to_csv(TRAIN_FILE, index=False, header=False)
valid_df.to_csv(VALID_FILE, index=False, header=False)
test_df.to_csv(TEST_FILE, index=False, header=False)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

# Upload processed files to S3 processed/
train_s3_uri = session.upload_data(TRAIN_FILE, bucket=BUCKET, key_prefix=f"{PROCESSED_PREFIX}/train")
valid_s3_uri = session.upload_data(VALID_FILE, bucket=BUCKET, key_prefix=f"{PROCESSED_PREFIX}/validation")
test_s3_uri  = session.upload_data(TEST_FILE,  bucket=BUCKET, key_prefix=f"{PROCESSED_PREFIX}/test")

print("Train S3 URI:", train_s3_uri)
print("Validation S3 URI:", valid_s3_uri)
print("Test S3 URI:", test_s3_uri)


Train shape: (139190, 12)
Validation shape: (29827, 12)
Test shape: (29827, 12)


Train S3 URI: s3://finalproject-fraud-detection/processed/train/train.csv
Validation S3 URI: s3://finalproject-fraud-detection/processed/validation/validation.csv
Test S3 URI: s3://finalproject-fraud-detection/processed/test/test.csv


In [0]:
# Create and launch the SageMaker built-in XGBoost training job
# A smaller instance type is used to reduce resource consumption

from sagemaker.estimator import Estimator

xgboost_container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1"
)

job_name = f"fraud-xgboost-{int(time.time())}"

xgb = Estimator(
    image_uri=xgboost_container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size=10,
    max_run=3600,
    output_path=f"s3://{BUCKET}/{MODEL_PREFIX}/training-output",
    sagemaker_session=session
)

xgb.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=120,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=1,
    verbosity=1
)

train_input = TrainingInput(
    s3_data=train_s3_uri,
    content_type="text/csv"
)

validation_input = TrainingInput(
    s3_data=valid_s3_uri,
    content_type="text/csv"
)

xgb.fit(
    inputs={
        "train": train_input,
        "validation": validation_input
    },
    job_name=job_name,
    wait=True
)

print("Training job completed.")
print("Model artifact:", xgb.model_data)


2026-03-13 23:01:39 Starting - Starting the training job.

.

.


2026-03-13 23:02:08 Starting - Preparing the instances for training.

.

...

.


2026-03-13 23:04:30 Training - Training image download completed. Training in progress..

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-13 23:04:32.130 ip-10-0-133-161.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-13 23:04:32.203 ip-10-0-133-161.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-13:23:04:32:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-13:23:04:32:INFO] Failed to parse hyperparameter eval_metric value auc to Json.
Returning the value itself
[2026-03-13:23:04:32:INFO] Failed to parse hyperparameter objective value binary:logistic to Json.
Returning the value itself
[2026-03-13:23:04:32:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-13:23:04:32:INFO] Run


2026-03-13 23:05:09 Uploading - Uploading generated training model
2026-03-13 23:05:09 Completed - Training job completed


Training seconds: 155
Billable seconds: 155
Training job completed.
Model artifact: s3://finalproject-fraud-detection/model/training-output/fraud-xgboost-1773442896/output/model.tar.gz


In [0]:
# Copy the trained model artifact to the required S3 model folder

from urllib.parse import urlparse

model_artifact_uri = xgb.model_data
parsed = urlparse(model_artifact_uri)

source_bucket = parsed.netloc
source_key = parsed.path.lstrip("/")

target_key = f"{MODEL_PREFIX}/xgboost/model-{int(time.time())}.tar.gz"

copy_source = {
    "Bucket": source_bucket,
    "Key": source_key
}

s3_client.copy_object(
    Bucket=BUCKET,
    CopySource=copy_source,
    Key=target_key
)

final_model_s3_uri = f"s3://{BUCKET}/{target_key}"
print("Copied model artifact to:", final_model_s3_uri)


Copied model artifact to: s3://finalproject-fraud-detection/model/xgboost/model-1773443126.tar.gz


In [0]:
# Deploy the trained model as a real-time SageMaker endpoint

predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer()
)

endpoint_name = predictor.endpoint_name
print("Endpoint deployed successfully.")
print("Endpoint name:", endpoint_name)


-

-

-

-

-

-

!

Endpoint deployed successfully.
Endpoint name: sagemaker-xgboost-2026-03-13-23-05-26-528


In [0]:
# =========================================================
# Evaluate the deployed model on the test set
# Metrics: Accuracy, Precision, Recall, F1-score, ROC-AUC
# =========================================================

# Separate features and labels
X_test_only = test_df.iloc[:, 1:].reset_index(drop=True)
y_test_only = test_df.iloc[:, 0].reset_index(drop=True)

# Mini-batch inference to avoid large request payloads
BATCH_SIZE = 500
pred_probs = []

for start in range(0, len(X_test_only), BATCH_SIZE):
    batch = X_test_only.iloc[start:start + BATCH_SIZE]

    payload = "\n".join(
        ",".join(map(str, row))
        for row in batch.to_numpy()
    )

    response = predictor.predict(payload)

    # Built-in XGBoost endpoint returns probabilities
    # {"predictions": [{"score": 0.1}, {"score": 0.9}, ...]}
    batch_scores = [item["score"] for item in response["predictions"]]
    pred_probs.extend(batch_scores)

pred_probs = np.array(pred_probs)

# Convert probabilities to labels using threshold 0.5
pred_labels = (pred_probs >= 0.5).astype(int)

# =========================================================
# Compute evaluation metrics
# =========================================================

accuracy = accuracy_score(y_test_only, pred_labels)
precision = precision_score(y_test_only, pred_labels)
recall = recall_score(y_test_only, pred_labels)
f1 = f1_score(y_test_only, pred_labels)
roc_auc = roc_auc_score(y_test_only, pred_probs)

# =========================================================
# Print results
# =========================================================

print("Evaluation Metrics")
print("------------------")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_only, pred_labels))

print("\nClassification Report:")
print(classification_report(y_test_only, pred_labels, digits=4))


Evaluation Metrics
------------------
Accuracy  : 0.9980
Precision : 0.9718
Recall    : 0.9789
F1-score  : 0.9753
ROC-AUC   : 0.9992

Confusion Matrix:
[[28560    35]
 [   26  1206]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9991    0.9988    0.9989     28595
           1     0.9718    0.9789    0.9753      1232

    accuracy                         0.9980     29827
   macro avg     0.9854    0.9888    0.9871     29827
weighted avg     0.9980    0.9980    0.9980     29827



In [0]:
# Test one sample record from the test set to verify endpoint inference

sample_features = X_test_only.iloc[0]
sample_true_label = int(y_test_only.iloc[0])

sample_payload = ",".join(map(str, sample_features.to_list()))
sample_response = predictor.predict(sample_payload)

sample_score = sample_response["predictions"][0]["score"]
sample_pred = 1 if sample_score >= 0.5 else 0

print("Sample True Label:", sample_true_label)
print("Sample Predicted Score:", round(sample_score, 6))
print("Sample Predicted Label:", sample_pred)


Sample True Label: 0
Sample Predicted Score: 3.6e-05
Sample Predicted Label: 0


In [0]:
# Save a small deployment verification result to S3 predictions/
# This is useful for later Athena / QuickSight testing

sample_result_df = pd.DataFrame([
    {
        "true_label": sample_true_label,
        "predicted_score": float(sample_score),
        "predicted_label": int(sample_pred),
        "endpoint_name": endpoint_name,
        "prediction_time": pd.Timestamp.utcnow().isoformat()
    }
])

sample_result_path = "/tmp/sample_prediction_result.csv"
sample_result_df.to_csv(sample_result_path, index=False)

sample_prediction_s3_uri = session.upload_data(
    sample_result_path,
    bucket=BUCKET,
    key_prefix="predictions/offline-endpoint-check"
)

print("Sample prediction result uploaded to:", sample_prediction_s3_uri)
sample_result_df


Sample prediction result uploaded to: s3://finalproject-fraud-detection/predictions/offline-endpoint-check/sample_prediction_result.csv


,true_label,predicted_score,predicted_label,endpoint_name,prediction_time
0,0,0.000036,0,sagemaker-xgboost-2026-03-13-23-05-26-528,2026-03-13T23:09:03.528026+00:00


In [0]:
import boto3
import json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

bucket = "finalproject-fraud-detection"
prefix = "predictions/realtime/"

s3 = boto3.client("s3")

rows = []

resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in resp.get("Contents", []):
    
    body = s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read()
    record = json.loads(body.decode())

    rows.append(record)

df = pd.DataFrame(rows)

y_true = df["actual_isFraud"].astype(int)
y_pred = df["predicted_label"].astype(int)
y_score = df["predicted_score"].astype(float)

print("Accuracy :", accuracy_score(y_true,y_pred))
print("Precision:", precision_score(y_true,y_pred))
print("Recall   :", recall_score(y_true,y_pred))
print("F1       :", f1_score(y_true,y_pred))
print("ROC AUC  :", roc_auc_score(y_true,y_score))

print(df["actual_isFraud"].value_counts(dropna=False))
print(df["predicted_label"].value_counts(dropna=False))



Accuracy : 1.0
Precision: 0.0
Recall   : 0.0
F1       : 0.0
ROC AUC  : nan
actual_isFraud
0    80
Name: count, dtype: int64
predicted_label
0    80
Name: count, dtype: int64


/sagemaker_packages/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/sagemaker_packages/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/sagemaker_packages/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/sagemaker_packag